# Signed IFFT color + continuous amplitude alpha
当前默认数值0→max，各图最终端点未定；white_max=0。

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline import *
F, metadata = load_image(ROOT/"data/input_ifft.tif")
cfg=json.loads((ROOT/"configs/default.json").read_text())
print(metadata)

## 1. Signed灰度显示窗口与D配色
默认全图0.5/99.5百分位窗口；标量gamma=1，随后分段green-white LUT。两端截断并量化；原始数据不变。

In [ ]:
L, bounds, clipped = prepare_ifft_display(F,cfg)
green, _, _ = render_layers(L, np.ones(F.shape,dtype=bool),cfg)
plt.figure(figsize=(8,8)); plt.imshow(green, interpolation="nearest"); plt.axis("off");
print("Display bounds:",bounds,"clipped:",clipped)

## 2. 原始IFFT取绝对值，再Gaussian sigma=5 px
注意这里用F，不是绿色通道L。此指标仅用于区域选择，不替代signed条纹。

In [ ]:
absolute, smooth = amplitude_map(F, sigma=cfg.get("sigma_px",5))
plt.figure(figsize=(8,8)); plt.imshow(smooth,cmap="magma"); plt.colorbar();

## 3. 连续alpha
端点在平滑幅度S上计算，数值0→max。可修改cfg["alpha"]的full模式/数值；这是显示权重，不是晶界置信度。

In [ ]:
alpha, endpoints = choose_alpha(smooth,cfg)
# 例如最亮5%完全显示：cfg["alpha"]["full"]={"mode":"percentile","value":95}
# 修改后重新运行本单元格。
plt.figure(figsize=(8,8)); plt.imshow(alpha,cmap="gray",vmin=0,vmax=1);
print(endpoints)


## 4. 连续alpha应用回第一步RGB
Gaussian不用于平滑显示的条纹；RGB与alpha独立。

In [ ]:
rgb, rgba, black = render_layers(L,alpha,cfg)
plt.figure(figsize=(8,8)); plt.imshow(black,interpolation="nearest"); plt.axis("off");


## 5. 保存新的完整运行记录
修改run名称后运行，避免覆盖旧版本。可另加tem_path，并在cfg中设置TEM显示窗口。

In [ ]:
cfg["alpha"]=effective_alpha_config(endpoints)
if endpoints["mode"]=="binary": cfg["threshold"]={"mode":"manual","value":endpoints["threshold"]}
# manifest=run_pipeline(ROOT/"data/input_ifft.tif", ROOT/"runs/notebook_run_003",cfg)
